# Combined RRF 颱風路徑相似度 — 最終優化版（含降水訊號）

三路（+可選降水）排名融合（Reciprocal Rank Fusion）預測颱風侵臺路徑類型（1–9 類）。

**LOO-CV 準確率：79.8%（158/198，Cat 1–9，未啟用降水）**

```
query typhoon
    ├── KNN  (11-dim summary features)  → rank_knn
    ├── DTW  (4-dim time series, 500km) → rank_dtw
    ├── Rule (CWA geometric path)       → rank_rule
    └── Rainfall (event_rain[region])   → rank_rain   ← 可選
                    ↓
score = α/(k+rank_knn) + w_dtw/(k+rank_dtw) + w_rule/(k+rank_rule) + w_rain/(k+rank_rain)
```

本 notebook 為**完全獨立**版本：所有演算法內嵌，僅需標準科學套件，
資料統一讀取 `typhoons_overview.json`（軌跡於 `path.position_intensity`，
事件降水於 `event_rain_tn`/`event_rain_kh`）。

## 0. Imports

In [27]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from sklearn.preprocessing import StandardScaler

## 1. 配置與降水地區

降水地區集中定義於 `RAINFALL_REGIONS`，新增地區只需加入一筆（code → overview 欄位 + 顯示名）。

| 參數 | 值 | 說明 |
|------|-----|------|
| `alpha` | 0.10 | KNN 排名權重 |
| `rule_weight` | 0.40 | Rule-Based 排名權重 |
| `rrf_k` | 30 | RRF 平滑常數 |
| `k` | 5 | top-k 類比颱風 |
| `dtw_weights` | [1.0, 0.5, 1.0, 0.5] | DTW 各維權重 [r, θ, wind, pressure] |
| `feature_weights` | [3.0, 2.0, 0.5×6, 2.5, 0.5, 0.5] | KNN 特徵權重 |
| `use_rainfall` | False | 是否啟用降水訊號 |
| `rainfall_region` | tn | 降水地區（tn=臺南、kh=高雄） |
| `rainfall_weight` | 0.15 | 啟用時降水訊號權重（DTW 讓出） |

In [28]:
# 資料路徑：請調整為實際 preprocessed 目錄
PROCESSED_DIR = "../data/typhoon/preprocessed"

# 降水地區設定（可擴充）
RAINFALL_REGIONS = {
    "tn": {"field": "event_rain_tn", "label": "臺南"},
    "kh": {"field": "event_rain_kh", "label": "高雄"},
}
DEFAULT_REGION = "tn"

CONFIG = {
    "alpha": 0.10,
    "rule_weight": 0.40,
    "k": 5,
    "rrf_k": 30,
    "pool_size_factor": 10,
    "impact_radius_km": 500.0,
    "dtw_weights": [1.0, 0.5, 1.0, 0.5],
    "feature_weights": [3.0, 2.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 2.5, 0.5, 0.5],
    "use_rainfall": False,
    "rainfall_region": "tn",
    "rainfall_weight": 0.15,
    "valid_categories": ["1", "2", "3", "4", "5", "6", "7", "8", "9"],
}

## 2. 資料結構

In [29]:
@dataclass
class TyphoonRecord:
    typhoon_id: str
    year: int
    name_zh: str
    name_en: str
    taiwan_track_category: str
    birth_lon: Optional[float]
    birth_lat: Optional[float]
    landfall_location: Optional[str]
    event_rain: dict           # {region_code: mm|None}
    track: pd.DataFrame = field(repr=False)

    def get_rainfall(self, region):
        return self.event_rain.get(region)


@dataclass
class TyphoonFeatures:
    typhoon_id: str
    min_distance_to_taiwan: float
    mean_angle: float
    max_wind_kt: float
    max_wind_in_window_kt: float
    approach_speed_kmh: float
    min_pressure_mb: float
    intensification_rate: float
    rain_proxy: float
    is_landfall: bool
    birth_lon: float
    birth_lat: float
    impact_window_r: np.ndarray = field(repr=False)
    impact_window_theta: np.ndarray = field(repr=False)
    impact_window_wind: np.ndarray = field(repr=False)
    impact_window_pressure: np.ndarray = field(repr=False)

    def to_feature_vector(self):
        return np.array([
            self.min_distance_to_taiwan, self.mean_angle, self.max_wind_kt,
            self.max_wind_in_window_kt, self.approach_speed_kmh, self.min_pressure_mb,
            self.intensification_rate, self.rain_proxy, float(self.is_landfall),
            self.birth_lon, self.birth_lat,
        ], dtype=np.float64)

    def get_impact_window_matrix(self):
        return np.column_stack([
            self.impact_window_r, self.impact_window_theta,
            self.impact_window_wind, self.impact_window_pressure,
        ])


@dataclass
class SimilarityResult:
    query_id: str
    similar_ids: list
    distances: list
    scores: list

## 3. 資料載入（統一資料源 typhoons_overview.json）

過濾條件：有路徑分類（1–9 或「特殊」）且具軌跡點 → 共 207 筆評估池。

In [30]:
_RAIN_NA = ("", "---", "nan", "none", "na", "n/a")


def _parse_rain(val):
    if val is None:
        return None
    if isinstance(val, (int, float)):
        v = float(val)
        return v if not np.isnan(v) else None
    s = str(val).strip()
    if s.lower() in _RAIN_NA:
        return None
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group()) if m else None


class DataLoader:
    def __init__(self, processed_dir):
        self.processed_dir = Path(processed_dir)
        self._records = []
        self._index = {}

    @property
    def records(self):
        return self._records

    def load(self):
        path = self.processed_dir / "typhoons_overview.json"
        with open(path, "r", encoding="utf-8") as f:
            dataset = json.load(f)
        typhoons = dataset["typhoons"] if isinstance(dataset, dict) else dataset

        for t in typhoons:
            category = str(t.get("taiwan_track_category", "") or "").strip()
            points = (t.get("path") or {}).get("position_intensity") or []
            if not category or len(points) < 2:
                continue
            track_df = pd.DataFrame(points)
            if "timestamp_utc" in track_df.columns:
                track_df["timestamp_utc"] = pd.to_datetime(
                    track_df["timestamp_utc"], errors="coerce", utc=True
                )
            event_rain = {
                code: _parse_rain(t.get(cfg["field"]))
                for code, cfg in RAINFALL_REGIONS.items()
            }
            rec = TyphoonRecord(
                typhoon_id=t["typhoon_id"], year=int(t["year"]),
                name_zh=t.get("name_zh", ""), name_en=t.get("name_en", ""),
                taiwan_track_category=category,
                birth_lon=t.get("genesis_longitude"), birth_lat=t.get("genesis_latitude"),
                landfall_location=t.get("landfall_location"),
                event_rain=event_rain, track=track_df,
            )
            self._records.append(rec)
            self._index[rec.typhoon_id] = rec
        print(f"載入 {len(self._records)} 筆颱風（來源 typhoons_overview.json）")
        return self

    def get(self, tid):
        return self._index[tid]

    def get_all_ids(self):
        return [r.typhoon_id for r in self._records]

## 4. 特徵提取

11 維摘要特徵 + 500km impact window 內 4 維時序路徑。座標修正 `dx*=cos(lat)`；
迎風面 rain proxy；接近速度只計 r<500km 段。

In [31]:
TAIWAN_LAT, TAIWAN_LON = 23.7, 121.0
EARTH_RADIUS_KM = 6371.0
TAIWAN_NORMAL_THETA_RAD = np.radians(60.0)


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(a))


def haversine_vec(lats, lons, ref_lat=TAIWAN_LAT, ref_lon=TAIWAN_LON):
    lat1, lon1 = np.radians(lats), np.radians(lons)
    lat2, lon2 = np.radians(ref_lat), np.radians(ref_lon)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(a))


def polar_coordinates(lats, lons):
    r = haversine_vec(lats, lons)
    dy = lats - TAIWAN_LAT
    dx = (lons - TAIWAN_LON) * np.cos(np.radians(lats))
    return r, np.arctan2(dy, dx)


class TyphoonFeatureExtractor:
    def __init__(self, impact_radius_km=500.0):
        self.impact_radius_km = impact_radius_km

    def extract(self, typhoon_id, track, birth_lon=None, birth_lat=None, landfall_location=None):
        track = track.copy()
        lats = track["latitude"].values.astype(float)
        lons = track["longitude"].values.astype(float)
        r, theta = polar_coordinates(lats, lons)
        winds = track["wind_kt"].fillna(0).values.astype(float)
        pressures = track["pressure_mb"].fillna(1013).values.astype(float)

        in_window = r < self.impact_radius_km
        if in_window.sum() < 2:
            nearest = np.argsort(r)[: max(5, len(r) // 5)]
            in_window = np.zeros(len(r), dtype=bool)
            in_window[nearest] = True

        wr, wt, ww, wp = r[in_window], theta[in_window], winds[in_window], pressures[in_window]
        mean_angle = float(np.arctan2(np.mean(np.sin(wt)), np.mean(np.cos(wt))))
        max_wind = float(np.max(winds)) if len(winds) else 0.0
        max_wind_w = float(np.max(ww)) if len(ww) else 0.0
        approach = self._speed(track, in_window)
        valid_p = pressures[pressures < 1013]
        min_p = float(np.min(valid_p)) if len(valid_p) else 1013.0
        inten = self._intensification(winds, in_window)
        safe_r = np.maximum(wr, 1.0)
        wind_dir = np.maximum(0, np.cos(wt - TAIWAN_NORMAL_THETA_RAD))
        rain_proxy = float(np.mean(ww * (0.5 + 0.5 * wind_dir) / safe_r))
        is_lf = landfall_location is not None and str(landfall_location).strip() not in ("", "---", "nan", "None")

        return TyphoonFeatures(
            typhoon_id=typhoon_id, min_distance_to_taiwan=float(np.min(r)),
            mean_angle=mean_angle, max_wind_kt=max_wind, max_wind_in_window_kt=max_wind_w,
            approach_speed_kmh=approach, min_pressure_mb=min_p, intensification_rate=inten,
            rain_proxy=rain_proxy, is_landfall=is_lf,
            birth_lon=birth_lon if birth_lon is not None else float(lons[0]),
            birth_lat=birth_lat if birth_lat is not None else float(lats[0]),
            impact_window_r=wr, impact_window_theta=wt,
            impact_window_wind=ww, impact_window_pressure=wp,
        )

    def extract_all(self, loader):
        feats = {r.typhoon_id: self.extract(r.typhoon_id, r.track, r.birth_lon, r.birth_lat, r.landfall_location)
                 for r in loader.records}
        print(f"提取 {len(feats)} 筆特徵")
        return feats

    def _speed(self, track, in_window):
        if in_window.sum() < 2:
            return 0.0
        wt = track[in_window].reset_index(drop=True)
        la, lo = wt["latitude"].values, wt["longitude"].values
        dist = sum(haversine(la[i-1], lo[i-1], la[i], lo[i]) for i in range(1, len(la)))
        if "timestamp_utc" in wt.columns and pd.notna(wt["timestamp_utc"].iloc[0]):
            ts = pd.to_datetime(wt["timestamp_utc"])
            dt_h = (ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600
            if dt_h > 0:
                return dist / dt_h
        return dist / max((len(la) - 1) * 3, 1)

    def _intensification(self, winds, in_window):
        idx = np.where(in_window)[0]
        if len(idx) < 2:
            return 0.0
        half = max(2, len(idx) // 2)
        fw = winds[idx[:half]]
        return float(np.polyfit(np.arange(len(fw)), fw, 1)[0]) if len(fw) >= 2 else 0.0

## 5. KNN 相似度

In [32]:
class KNNSimilarity:
    def __init__(self, feature_weights=None):
        self.feature_weights = feature_weights
        self.scaler = StandardScaler()
        self._ids = []
        self._vectors = None

    def fit(self, feature_dict):
        self._ids = list(feature_dict.keys())
        raw = np.array([feature_dict[t].to_feature_vector() for t in self._ids])
        self._vectors = self.scaler.fit_transform(raw)
        if self.feature_weights is not None:
            self._vectors *= np.array(self.feature_weights, dtype=np.float64)

    def find_similar(self, query_id, k=5, exclude_self=True):
        idx = self._ids.index(query_id)
        d = np.linalg.norm(self._vectors - self._vectors[idx], axis=1)
        ids, dists = [], []
        for i in np.argsort(d):
            if exclude_self and self._ids[i] == query_id:
                continue
            ids.append(self._ids[i]); dists.append(float(d[i]))
            if len(ids) >= k:
                break
        md = max(dists) if dists else 1.0
        return SimilarityResult(query_id, ids, dists, [1 - x / (md + 1e-8) for x in dists])

    def transform_query(self, vec):
        s = self.scaler.transform(vec.reshape(1, -1))
        if self.feature_weights is not None:
            s *= np.array(self.feature_weights)
        return s.flatten()

    def find_similar_by_vector(self, query_vec, k=5):
        s = self.transform_query(query_vec)
        d = np.linalg.norm(self._vectors - s, axis=1)
        top = np.argsort(d)[:k]
        ids = [self._ids[i] for i in top]; dists = [float(d[i]) for i in top]
        md = max(dists) if dists else 1.0
        return SimilarityResult("query", ids, dists, [1 - x / (md + 1e-8) for x in dists])

## 6. DTW 時序相似度

環形方位角距離 + 物理標準化 + 距離衰減加權 `exp(-r/200)` + Sakoe-Chiba band(0.3)。

In [33]:
NORM_R, NORM_THETA, NORM_WIND, NORM_PRESSURE = 300.0, np.pi, 100.0, 50.0
SAKOE_CHIBA_RATIO = 0.3


def _circular(a, b):
    d = abs(a - b)
    return min(d, 2 * np.pi - d)


def _dtw(seq1, seq2, weights=None):
    n, m = len(seq1), len(seq2)
    if n == 0 or m == 0:
        return float("inf")
    if weights is None:
        weights = np.ones(seq1.shape[1])
    w = weights / weights.sum()
    band = max(1, int(SAKOE_CHIBA_RATIO * max(n, m)))
    cost = np.full((n + 1, m + 1), float("inf"))
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(max(1, i - band), min(m, i + band) + 1):
            d = sum(
                w[k] * (_circular(seq1[i-1, k], seq2[j-1, k]) ** 2 if k == 1
                        else (seq1[i-1, k] - seq2[j-1, k]) ** 2)
                for k in range(seq1.shape[1])
            )
            cost[i, j] = d + min(cost[i-1, j], cost[i, j-1], cost[i-1, j-1])
    return float(np.sqrt(cost[n, m] / (n + m)))


def _normalize_matrix(mat):
    nz = mat.copy().astype(float)
    nz[:, 0] /= NORM_R
    nz[:, 1] /= NORM_THETA
    nz[:, 2] /= NORM_WIND
    nz[:, 3] = (nz[:, 3] - 950) / NORM_PRESSURE
    nz *= np.exp(-mat[:, 0] / 200.0)[:, None]
    return nz


class DTWSimilarity:
    def __init__(self, dtw_weights=None):
        self.dtw_weights = np.array(dtw_weights) if dtw_weights is not None else np.array([1.0, 1.0, 1.0, 0.5])
        self._ids = []
        self._matrices = {}
        self._cache = {}

    def fit(self, feature_dict):
        self._ids = list(feature_dict.keys())
        for t in self._ids:
            self._matrices[t] = _normalize_matrix(feature_dict[t].get_impact_window_matrix())

    def _dist(self, a, b):
        key = tuple(sorted([a, b]))
        if key not in self._cache:
            self._cache[key] = _dtw(self._matrices[a], self._matrices[b], self.dtw_weights)
        return self._cache[key]

    def find_similar(self, query_id, k=5, exclude_self=True):
        d = {t: self._dist(query_id, t) for t in self._ids if not (exclude_self and t == query_id)}
        top = sorted(d.items(), key=lambda x: x[1])[:k]
        ids = [x[0] for x in top]; dists = [x[1] for x in top]
        md = max(dists) if dists else 1.0
        return SimilarityResult(query_id, ids, dists, [1 - x / (md + 1e-8) for x in dists])

    def find_similar_by_matrix(self, query_mat, k=5):
        qn = _normalize_matrix(query_mat)
        d = {t: _dtw(qn, self._matrices[t], self.dtw_weights) for t in self._ids}
        top = sorted(d.items(), key=lambda x: x[1])[:k]
        ids = [x[0] for x in top]; dists = [x[1] for x in top]
        md = max(dists) if dists else 1.0
        return SimilarityResult("query", ids, dists, [1 - x / (md + 1e-8) for x in dists])

## 7. Rule-Based 路徑分類（CWA 幾何分類）

依優先序判斷：Cat6 東岸北上 → Cat8 南海北行 → 登陸文字解析(2/3/4/7) → Cat7 西側北上 → Cat1/5 海面通過 → 緯度判斷。

In [34]:
TAIWAN_CENTER_LAT = 23.5
TAIWAN_EAST_LON = 121.8
CONTEXT_RADIUS_KM = 500
LANDFALL_KM = 100
WEST_CROSSING_LON = 120.3

EAST_NORTH_KW = ["基隆","宜蘭","彭佳嶼","蘇澳","頭城","三貂角","新北","淡水","南澳","蘭陽","秀林"]
EAST_CENTRAL_KW = ["花蓮","新港","成功","秀姑巒","豐濱","長濱","靜浦","東澳","東河"]
EAST_SOUTH_KW = ["臺東","台東","大武","太麻里","滿州","鵝鑾鼻"]
WEST_SOUTH_KW = ["金門","高雄","小港","東石","台中","臺中","苗栗","彰化","雲林","嘉義"]
SOUTH_TIP_KW = ["恆春","屏東","楓港","枋寮"]


def _parse_landfall(loc):
    if not loc:
        return None
    loc = str(loc).strip()
    if loc in ("", "---", "nan", "None", "無登陸"):
        return None
    hh, hy = "花蓮" in loc, "宜蘭" in loc
    ht = "臺東" in loc or "台東" in loc
    hc = "成功" in loc or "新港" in loc
    hf = "豐濱" in loc or "長濱" in loc or "靜浦" in loc
    if hy and hh: return "east_north"
    if hy: return "east_north"
    if hh and hc: return "east_central"
    if hh and ht: return "east_central"
    if hh and hf: return "east_central"
    if hf: return "east_central"
    if hc: return "east_central"
    if hh: return "east_central"
    if "秀林" in loc: return "east_central"
    for kw in EAST_SOUTH_KW:
        if kw in loc: return "east_south"
    for kw in SOUTH_TIP_KW:
        if kw in loc: return "south_tip"
    for kw in WEST_SOUTH_KW:
        if kw in loc: return "west_south"
    for kw in EAST_NORTH_KW:
        if kw in loc: return "east_north"
    for kw in EAST_CENTRAL_KW:
        if kw in loc: return "east_central"
    return None


def _ctx_heading(lats, lons, ctx):
    cl, co = lats[ctx], lons[ctx]
    if len(cl) < 2:
        return 0.0
    dlat = cl[-1] - cl[0]
    dlon = (co[-1] - co[0]) * np.cos(np.radians(cl[0]))
    return float(np.degrees(np.arctan2(dlat, dlon)))


def _crossed_west(lats, lons, ci):
    s = max(0, ci - 3)
    pl, po = lats[s:], lons[s:]
    pd_ = haversine_vec(pl, po)
    near = pd_ < 400
    return bool(np.any(po[near] < WEST_CROSSING_LON)) if near.sum() else False


def _approach(lats, lons, ci):
    end, start = ci, max(0, ci - 8)
    if end - start < 2:
        start = max(0, ci - 3); end = min(len(lats) - 1, ci + 3)
    if end <= start:
        return 0.0
    dlats = np.diff(lats[start:end+1])
    dlons = np.diff(lons[start:end+1]) * np.cos(np.radians(lats[start:end+1][:-1]))
    return float(np.degrees(np.arctan2(np.mean(dlats), np.mean(dlons))))


def _R(cat, conf, reason):
    return {"predicted_category": cat, "confidence": conf, "reasoning": reason}


def classify_typhoon_by_rules(track, landfall_location=None):
    lats = track["latitude"].values.astype(float)
    lons = track["longitude"].values.astype(float)
    dist = haversine_vec(lats, lons)
    mind = float(np.min(dist))
    ci = int(np.argmin(dist))
    clat, clon = float(lats[ci]), float(lons[ci])
    ctx = dist < CONTEXT_RADIUS_KM
    if ctx.sum() < 2:
        return _R("1" if clat > TAIWAN_CENTER_LAT else "5", 0.4, "海面遠距通過")
    cl, co = lats[ctx], lons[ctx]
    entry_lat, exit_lat = float(cl[0]), float(cl[-1])
    ch = _ctx_heading(lats, lons, ctx)
    ap = _approach(lats, lons, ci)
    cw = _crossed_west(lats, lons, ci)
    has_lf = landfall_location is not None and str(landfall_location).strip() not in ("", "---", "nan", "None", "無登陸")
    lf = _parse_landfall(landfall_location) if has_lf else None
    conf = max(0.4, min(0.95, 1.0 - mind / 500))

    if not cw and clon > 120.8 and 50 < ch < 125 and clat < 26.0:
        er = np.sum(co > TAIWAN_EAST_LON - 0.5) / len(co)
        if er > 0.3:
            skip = ((ap > 145 and mind < 60)
                    or (has_lf and lf == "east_north" and mind < 80)
                    or (has_lf and lf == "east_central" and mind < 50 and ap > 120)
                    or (clat > 25.0 and mind > 200))
            if not skip:
                return _R("6", conf, "沿東岸北上")
    if entry_lat < 22.5 and ch < 80 and not cw and clon > 120.5 and clat < 24.0 and mind > 60:
        return _R("8", conf, "南部海面向北")
    if has_lf and lf:
        if lf == "east_north": return _R("2", conf, "登陸北部")
        if lf == "east_central": return _R("3", conf, "登陸中部")
        if lf == "east_south": return _R("4", conf, "登陸南部東岸")
        if lf == "west_south":
            if entry_lat < 22.5 and ch > 60 and exit_lat > entry_lat + 2.5:
                return _R("7", conf, "南方經西岸北上")
            return _R("9", conf * 0.7, "西岸登陸不規則")
        if lf == "south_tip":
            if entry_lat < 22.5 and ch > 60 and exit_lat > entry_lat + 3.0:
                return _R("7", conf, "南方經南端北上")
            return _R("4", conf * 0.8, "通過南端")
    if entry_lat < 22.0 and clon < 121.0 and 70 < ch < 140 and exit_lat > entry_lat + 3.0:
        return _R("7", conf, "南方沿西側北上")
    if has_lf:
        if clat >= 24.0: return _R("2", conf * 0.8, "登陸緯度判北部")
        if clat <= 22.8: return _R("4", conf * 0.8, "登陸緯度判南部")
        return _R("3", conf * 0.8, "登陸緯度判中部")
    if mind > LANDFALL_KM:
        return _R("1" if clat > TAIWAN_CENTER_LAT else "5", conf, "海面通過")
    if clat >= 24.0: return _R("2", conf * 0.7, "緯度判北部")
    if clat <= 22.5: return _R("4", conf * 0.7, "緯度判南部")
    return _R("3", conf * 0.7, "緯度判中部")

## 8. Combined RRF 融合（含降水訊號）

$$score(c)=\frac{\alpha}{k+r_{knn}}+\frac{w_{dtw}}{k+r_{dtw}}+\frac{w_{rule}}{k+r_{rule}}+\underbrace{\frac{w_{rain}}{k+r_{rain}}}_{use\_rainfall}$$

降水排名：依與 query 在指定地區降水量的差距排序（差距小→排前），無資料者排最後。
`use_rainfall=False` 時退化為原三訊號版（= 79.8% 基準）。

In [35]:
class CombinedSimilarity:
    def __init__(self, alpha=0.2, rule_weight=0.5, feature_weights=None, dtw_weights=None,
                 pool_size_factor=10, rrf_k=60,
                 use_rainfall=False, rainfall_region="tn", rainfall_weight=0.15):
        self.alpha = alpha
        self.rule_weight = rule_weight
        self.pool_size_factor = pool_size_factor
        self.rrf_k = rrf_k
        self.use_rainfall = use_rainfall
        self.rainfall_region = rainfall_region
        self.rainfall_weight = rainfall_weight if use_rainfall else 0.0
        self.knn = KNNSimilarity(feature_weights=feature_weights)
        self.dtw = DTWSimilarity(dtw_weights=dtw_weights)
        self._ids = []
        self._rule = {}
        self._rain = {}
        self._loader = None

    def fit(self, feature_dict, loader=None):
        self._ids = list(feature_dict.keys())
        self._loader = loader
        self.knn.fit(feature_dict)
        self.dtw.fit(feature_dict)
        if loader:
            for t in self._ids:
                rec = loader.get(t)
                self._rule[t] = classify_typhoon_by_rules(rec.track, rec.landfall_location)["predicted_category"]
                self._rain[t] = rec.get_rainfall(self.rainfall_region)
        wd = max(0.0, 1 - self.alpha - self.rule_weight - self.rainfall_weight)
        rm = f", Rain[{self.rainfall_region}]={self.rainfall_weight:.2f}" if self.use_rainfall else ""
        print(f"Combined RRF 擬合 {len(self._ids)} 筆 | KNN={self.alpha:.2f}, DTW={wd:.2f}, Rule={self.rule_weight:.2f}{rm}")

    def configure_rainfall(self, use_rainfall, region=None, weight=None):
        self.use_rainfall = use_rainfall
        if region is not None:
            self.rainfall_region = region
        self.rainfall_weight = (weight if weight is not None else self.rainfall_weight) if use_rainfall else 0.0
        if use_rainfall and self._loader:
            self._rain = {t: self._loader.get(t).get_rainfall(self.rainfall_region) for t in self._ids}

    def _rain_ranks(self, qr):
        if qr is None or not self._rain:
            return {}
        scored, missing = [], []
        for t in self._ids:
            r = self._rain.get(t)
            (missing if r is None else scored).append(t if r is None else (t, abs(qr - r)))
        scored.sort(key=lambda x: x[1])
        ranks = {t: i for i, (t, _) in enumerate(scored)}
        for i, t in enumerate(missing, len(scored)):
            ranks[t] = i
        return ranks

    def _fuse(self, knn_ranks, dtw_ranks, rule_ranks, rain_ranks, candidates, k, pool):
        w_knn, w_rule = self.alpha, self.rule_weight
        w_rain = self.rainfall_weight if rain_ranks else 0.0
        w_dtw = max(0.0, 1 - self.alpha - self.rule_weight - w_rain)
        out = {}
        for t in candidates:
            s = (w_knn / (self.rrf_k + knn_ranks.get(t, pool))
                 + w_dtw / (self.rrf_k + dtw_ranks.get(t, pool))
                 + w_rule / (self.rrf_k + rule_ranks.get(t, len(self._ids))))
            if w_rain > 0:
                s += w_rain / (self.rrf_k + rain_ranks.get(t, len(self._ids)))
            out[t] = s
        top = sorted(out.items(), key=lambda x: x[1], reverse=True)[:k]
        ids = [x[0] for x in top]; sc = [x[1] for x in top]
        ms = max(sc) if sc else 1.0
        norm = [x / (ms + 1e-8) for x in sc]
        return SimilarityResult("query", ids, [1 - x for x in norm], norm)

    def find_similar(self, query_id, k=5, exclude_self=True):
        pool = min(len(self._ids) - 1, k * self.pool_size_factor)
        kr = self.knn.find_similar(query_id, pool, exclude_self)
        dr = self.dtw.find_similar(query_id, pool, exclude_self)
        knn_ranks = {t: i for i, t in enumerate(kr.similar_ids)}
        dtw_ranks = {t: i for i, t in enumerate(dr.similar_ids)}
        qcat = self._rule.get(query_id, "")
        same = [t for t in self._ids if t != query_id and self._rule.get(t) == qcat]
        diff = [t for t in self._ids if t != query_id and self._rule.get(t) != qcat]
        rule_ranks = {t: i for i, t in enumerate(same)}
        rule_ranks.update({t: i for i, t in enumerate(diff, len(same))})
        rain_ranks = self._rain_ranks(self._rain.get(query_id)) if self.use_rainfall else {}
        cand = set(kr.similar_ids) | set(dr.similar_ids) | set(same[:pool])
        if rain_ranks:
            cand |= set(sorted(rain_ranks, key=rain_ranks.get)[:pool])
        cand.discard(query_id)
        res = self._fuse(knn_ranks, dtw_ranks, rule_ranks, rain_ranks, cand, k, pool)
        return SimilarityResult(query_id, res.similar_ids, res.distances, res.scores)

    def find_similar_by_vector(self, query_vec, k=5, query_features=None, query_rainfall=None):
        pool = min(len(self._ids), k * self.pool_size_factor)
        kr = self.knn.find_similar_by_vector(query_vec, k=pool)
        knn_ranks = {t: i for i, t in enumerate(kr.similar_ids)}
        dtw_ranks = {}
        if query_features is not None:
            dr = self.dtw.find_similar_by_matrix(query_features.get_impact_window_matrix(), k=pool)
            dtw_ranks = {t: i for i, t in enumerate(dr.similar_ids)}
        rule_ranks = {}
        if self._rule and kr.similar_ids:
            proxy = self._rule.get(kr.similar_ids[0], "")
            same = [t for t in self._ids if self._rule.get(t) == proxy]
            diff = [t for t in self._ids if self._rule.get(t) != proxy]
            rule_ranks = {t: i for i, t in enumerate(same)}
            rule_ranks.update({t: i for i, t in enumerate(diff, len(same))})
        rain_ranks = self._rain_ranks(query_rainfall) if (self.use_rainfall and query_rainfall is not None) else {}
        cand = set(kr.similar_ids)
        if dtw_ranks:
            cand |= set(list(dtw_ranks.keys())[:pool])
        if rule_ranks:
            cand |= set(t for t in self._ids if rule_ranks.get(t, len(self._ids)) < pool)
        if rain_ranks:
            cand |= set(sorted(rain_ranks, key=rain_ranks.get)[:pool])
        return self._fuse(knn_ranks, dtw_ranks, rule_ranks, rain_ranks, cand, k, pool)

## 9. 類比投票與 LOO 評估

In [36]:
def weighted_vote(similar_ids, distances, label_dict):
    vw = {}
    for tid, dist in zip(similar_ids, distances):
        cat = label_dict.get(tid)
        if cat is None:
            continue
        vw[cat] = vw.get(cat, 0.0) + float(np.exp(-dist))
    if not vw:
        return None, 0.0, {}
    tot = sum(vw.values())
    probs = {c: w / tot for c, w in vw.items()}
    best = max(probs, key=probs.get)
    return best, probs[best], probs


def compute_accuracy(results):
    per = {}
    for r in results:
        c = r["true_category"]
        per.setdefault(c, {"correct": 0, "total": 0})
        per[c]["total"] += 1
        per[c]["correct"] += int(r["is_correct"])
    for v in per.values():
        v["accuracy"] = v["correct"] / v["total"]
    total = sum(v["total"] for v in per.values())
    correct = sum(v["correct"] for v in per.values())
    return {"accuracy": correct / total if total else 0.0, "correct": correct, "total": total, "per_category": per}


def run_loo(sim, loader, label_dict, k=5, valid_categories=None):
    valid_categories = valid_categories or [str(i) for i in range(1, 10)]
    results = []
    for tid in loader.get_all_ids():
        rec = loader.get(tid)
        if rec.taiwan_track_category not in valid_categories:
            continue
        s = sim.find_similar(tid, k=k, exclude_self=True)
        pred, conf, probs = weighted_vote(s.similar_ids, s.distances, label_dict)
        results.append({
            "typhoon_id": tid, "name_zh": rec.name_zh, "year": rec.year,
            "true_category": rec.taiwan_track_category, "predicted_category": pred,
            "confidence": round(conf, 4), "is_correct": pred == rec.taiwan_track_category,
            "category_votes": probs,
            "similar_typhoons": [
                {"typhoon_id": t, "name_zh": loader.get(t).name_zh, "year": loader.get(t).year,
                 "category": label_dict.get(t), "distance": round(d, 4)}
                for t, d in zip(s.similar_ids, s.distances)
            ],
        })
    return results

## 10. 執行（基準：未啟用降水）

In [37]:
loader = DataLoader(PROCESSED_DIR).load()
extractor = TyphoonFeatureExtractor(impact_radius_km=CONFIG["impact_radius_km"])
features = extractor.extract_all(loader)
label_dict = {r.typhoon_id: r.taiwan_track_category for r in loader.records}

combined = CombinedSimilarity(
    alpha=CONFIG["alpha"], rule_weight=CONFIG["rule_weight"],
    feature_weights=CONFIG["feature_weights"], dtw_weights=CONFIG["dtw_weights"],
    pool_size_factor=CONFIG["pool_size_factor"], rrf_k=CONFIG["rrf_k"],
    use_rainfall=CONFIG["use_rainfall"], rainfall_region=CONFIG["rainfall_region"],
    rainfall_weight=CONFIG["rainfall_weight"],
)
combined.fit(features, loader=loader)

載入 207 筆颱風（來源 typhoons_overview.json）
提取 207 筆特徵
Combined RRF 擬合 207 筆 | KNN=0.10, DTW=0.50, Rule=0.40


In [38]:
print("執行 Leave-One-Out 評估...")
results = run_loo(combined, loader, label_dict, k=CONFIG["k"], valid_categories=CONFIG["valid_categories"])
metrics = compute_accuracy(results)
print(f"準確率：{metrics['accuracy']:.1%} ({metrics['correct']}/{metrics['total']})")

執行 Leave-One-Out 評估...
準確率：79.8% (158/198)


## 11. 結果

In [39]:
rows = [{"類型": c, "正確": v["correct"], "總計": v["total"], "準確率": f"{v['accuracy']:.1%}"}
        for c, v in sorted(metrics["per_category"].items())]
rows.append({"類型": "合計", "正確": metrics["correct"], "總計": metrics["total"], "準確率": f"{metrics['accuracy']:.1%}"})
pd.DataFrame(rows).set_index("類型")

,正確,總計,準確率
類型,,,
1,23,23,100.0%
2,25,29,86.2%
3,24,30,80.0%
4,18,21,85.7%
5,29,30,96.7%
6,28,30,93.3%
7,6,11,54.5%
8,3,6,50.0%
9,2,18,11.1%


## 12. 降水訊號比較（可選）

啟用降水訊號後類比檢索改以「降水規模相近」為導向；對純路徑分類準確率略降屬預期。

In [40]:
combined_rain = CombinedSimilarity(
    alpha=CONFIG["alpha"], rule_weight=CONFIG["rule_weight"],
    feature_weights=CONFIG["feature_weights"], dtw_weights=CONFIG["dtw_weights"],
    pool_size_factor=CONFIG["pool_size_factor"], rrf_k=CONFIG["rrf_k"],
    use_rainfall=True, rainfall_region="tn", rainfall_weight=0.15,
)
combined_rain.fit(features, loader=loader)
res_rain = run_loo(combined_rain, loader, label_dict, k=CONFIG["k"], valid_categories=CONFIG["valid_categories"])
m_rain = compute_accuracy(res_rain)
print(f"未啟用降水：{metrics['accuracy']:.1%} ({metrics['correct']}/{metrics['total']})")
print(f"啟用降水(tn,0.15)：{m_rain['accuracy']:.1%} ({m_rain['correct']}/{m_rain['total']})")

Combined RRF 擬合 207 筆 | KNN=0.10, DTW=0.35, Rule=0.40, Rain[tn]=0.15
未啟用降水：79.8% (158/198)
啟用降水(tn,0.15)：78.3% (155/198)


## 13. 單筆預測範例（輸入 example → 生成指定預測結果）

`predict_example(...)` 支援兩種輸入：
- **既有颱風 ID**（如 `"201909"`）：以 LOO 方式排除自身後檢索類比。
- **自訂軌跡**（list of dict，需含 latitude/longitude，wind_kt/pressure_mb 選填）：模擬新颱風即時預測。

可選參數 `use_rainfall` / `rainfall_region` / `expected_rainfall` 啟用降水訊號。

In [41]:
def predict_example(query_id=None, track=None, expected_rainfall=None,
                    use_rainfall=False, rainfall_region="tn", k=5):
    """輸入一個 example，生成預測結果（分類 + 信心 + 類比颱風 + 降水推估）。"""
    combined.configure_rainfall(use_rainfall, rainfall_region, CONFIG["rainfall_weight"])

    if query_id is not None:
        rec = loader.get(query_id)
        sim = combined.find_similar(query_id, k=k, exclude_self=True)
        true_cat = rec.taiwan_track_category
        title = f"{rec.name_zh}（{rec.year}, {query_id}）"
    else:
        df = pd.DataFrame(track)
        if "timestamp_utc" not in df.columns:
            df["timestamp_utc"] = pd.date_range("2000-01-01", periods=len(df), freq="6h")
        qf = extractor.extract("query", df)
        sim = combined.find_similar_by_vector(
            qf.to_feature_vector(), k=k, query_features=qf,
            query_rainfall=expected_rainfall if use_rainfall else None,
        )
        true_cat = None
        title = "自訂軌跡（新颱風）"

    pred, conf, votes = weighted_vote(sim.similar_ids, sim.distances, label_dict)

    print("=" * 56)
    print(f"輸入：{title}")
    if use_rainfall:
        rl = RAINFALL_REGIONS[rainfall_region]["label"]
        print(f"降水訊號：啟用（地區={rl}/{rainfall_region}"
              + (f"，預期降水={expected_rainfall}mm" if expected_rainfall is not None else "") + "）")
    print(f"預測類型：Cat {pred}   信心度：{conf:.1%}"
          + (f"   真實類型：Cat {true_cat}   {'✓' if pred == true_cat else '✗'}" if true_cat else ""))
    print("\n類比颱風 Top-{}：".format(k))
    for t, d in zip(sim.similar_ids, sim.distances):
        ar = loader.get(t)
        print(f"  {ar.year} {ar.name_zh:8s} Cat {ar.taiwan_track_category}  dist={d:.4f}"
              f"  rain[{rainfall_region}]={ar.get_rainfall(rainfall_region)}")
    print("\n分類投票：")
    for c, p in sorted(votes.items(), key=lambda x: -x[1]):
        print(f"  Cat {c}: {p:.1%}  " + "█" * int(p * 20))

    # 降水推估（各地區，以類比颱風估計）
    print("\n降水推估（類比颱風）：")
    for code, cfg in RAINFALL_REGIONS.items():
        vals = [loader.get(t).get_rainfall(code) for t in sim.similar_ids]
        vals = [v for v in vals if v is not None]
        if vals:
            print(f"  {cfg['label']}({code}): mean={np.mean(vals):.1f}mm  "
                  f"median={np.median(vals):.1f}mm  min={min(vals):.1f}  max={max(vals):.1f}  n={len(vals)}")
    return {"predicted_category": pred, "confidence": conf, "votes": votes,
            "similar_ids": sim.similar_ids, "true_category": true_cat}

### 範例 A：既有颱風（LOO 檢索）

In [42]:
# 取一個 Cat 3 的既有颱風作為示範（可改成任何 loader 內的 typhoon_id）
example_id = next(r.typhoon_id for r in loader.records if r.taiwan_track_category == "3")
_ = predict_example(query_id=example_id, k=5)

輸入：溫妮（1958, 195807）
預測類型：Cat 3   信心度：100.0%   真實類型：Cat 3   ✓

類比颱風 Top-5：
  1986 艾貝       Cat 3  dist=0.0000  rain[tn]=137.7
  1997 安珀       Cat 3  dist=0.0320  rain[tn]=22.8
  1994 提姆       Cat 3  dist=0.1152  rain[tn]=44.6
  1959 瓊安       Cat 3  dist=0.1190  rain[tn]=99.8
  2016 梅姬       Cat 3  dist=0.1463  rain[tn]=363.0

分類投票：
  Cat 3: 100.0%  ████████████████████

降水推估（類比颱風）：
  臺南(tn): mean=133.6mm  median=99.8mm  min=22.8  max=363.0  n=5
  高雄(kh): mean=96.9mm  median=119.3mm  min=42.0  max=148.0  n=5


### 範例 B：自訂軌跡（新颱風）+ 啟用降水訊號

輸入一條西行穿越型路徑，並指定預期降水量 200mm（臺南），啟用降水排序。

In [43]:
custom_track = [
    {"latitude": 15.0, "longitude": 135.0, "wind_kt": 25, "pressure_mb": 1002},
    {"latitude": 18.0, "longitude": 129.0, "wind_kt": 50, "pressure_mb": 980},
    {"latitude": 21.0, "longitude": 124.0, "wind_kt": 75, "pressure_mb": 955},
    {"latitude": 23.0, "longitude": 121.0, "wind_kt": 80, "pressure_mb": 950},
    {"latitude": 25.5, "longitude": 119.5, "wind_kt": 55, "pressure_mb": 975},
]
_ = predict_example(track=custom_track, use_rainfall=True, rainfall_region="tn",
                    expected_rainfall=200, k=5)

輸入：自訂軌跡（新颱風）
降水訊號：啟用（地區=臺南/tn，預期降水=200mm）
預測類型：Cat 6   信心度：62.0%

類比颱風 Top-5：
  1959 芙瑞達      Cat 6  dist=0.0000  rain[tn]=26.7
  1961 貝蒂       Cat 6  dist=0.1401  rain[tn]=7.9
  1958 溫妮       Cat 3  dist=0.1814  rain[tn]=155.9
  1959 畢莉       Cat 1  dist=0.2226  rain[tn]=111.8
  1965 黛納       Cat 6  dist=0.2227  rain[tn]=10.2

分類投票：
  Cat 6: 62.0%  ████████████
  Cat 3: 19.4%  ███
  Cat 1: 18.6%  ███

降水推估（類比颱風）：
  臺南(tn): mean=62.5mm  median=26.7mm  min=7.9  max=155.9  n=5
  高雄(kh): mean=80.3mm  median=57.2mm  min=10.8  max=186.1  n=5
